In [1]:
import ee
import multiprocessing
from tqdm import tqdm
from retry import retry
import requests

In [2]:
ee.Authenticate()
ee.Initialize(project='ee-kerrilu')
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
# https://gorelick.medium.com/fast-er-downloads-a2abd512aa26

NAIP = ee.ImageCollection("USDA/NAIP/DOQQ")
NAIP = NAIP.filterDate('2021-01-01', '2024-01-01')
field_boundaries = ee.FeatureCollection("projects/ee-kerrilu/assets/US-fields/US_fields_2021_to_2023")
centroids = ee.FeatureCollection("projects/ee-kerrilu/assets/US-fields/centroids_8m_resolution")

@retry(tries=10, delay=1, backoff=2)
def getResult(enumerated_point):
    i, point = enumerated_point
    coord = ee.Geometry.Point(point['coordinates'])
    square = coord.buffer(2048, 0.001).bounds()

    images = NAIP.filterBounds(square).sort('system:time_start')
    image_mosaic = images.mosaic()
    image = image_mosaic.clip(square)

    most_recent_image = images.sort('system:time_start', False).first()
    image_date = ee.Number(most_recent_image.get('system:time_start'))

    field_boundaries_current = field_boundaries.filterBounds(square)

    def func_cvn(feature):
        feature_date = ee.Date.parse('yyyy-MM-dd HH:mm:ss', ee.String(feature.get('date')).slice(0, 19)).millis()
        time_diff_from_image = image_date.subtract(feature_date).abs()
        return(feature.set({'time_diff_from_image': time_diff_from_image}))

    field_boundaries_current = field_boundaries_current.map(func_cvn).sort('time_diff_from_image')

    # https://gis.stackexchange.com/questions/258866/deleting-close-points-from-featurecollection-using-buffer-in-google-eath-engine
    def process_current_polygon(current_polygon, current_list):
        current_list = ee.List(current_list)

        def func_omc(f):
            intersection_area =  f.intersection(current_polygon, 0.001).area()
            union_area = f.union(current_polygon).area()
            iou = intersection_area.divide(union_area)
            return(f.set({"iou":iou}))

        ious = ee.FeatureCollection(current_list).map(func_omc)

        high_ious = ious.filter(ee.Filter.gt('iou', 0.01))
        cond = high_ious.size().eq(0)
        return ee.Algorithms.If(cond, current_list.add(current_polygon), current_list)

    def filter_polygons(polygons):
        initial_list = ee.List([])
        polygons_to_keep = polygons.iterate(process_current_polygon, initial_list)
        polygons_to_keep = ee.FeatureCollection(ee.List(polygons_to_keep))
        return(polygons_to_keep)

    field_boundaries_current = filter_polygons(field_boundaries_current)

    def func_ywt(feature):
        return feature.intersection(square, 0.001)

    field_boundaries_current = field_boundaries_current.map(func_ywt)

    masks = field_boundaries_current.reduceToImage(
        properties=['id'],
        reducer=ee.Reducer.first()
    )
    mask = masks.clip(square)

    filename = f'training_data/8m-resolution/US_NAIP_Semios/images/{(i+1):07d}.tif'
    url = image.getDownloadURL({
        'region':square,
        'dimensions':'512x512',
        'crs':'EPSG:4326',
        'format': 'GEO_TIFF'
    })
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        raise r.raise_for_status()
    with open(filename, 'wb') as out_file:
        out_file.write(r.content)

    filename = f'training_data/8m-resolution/US_NAIP_Semios/masks/{(i+1):07d}.tif'
    url = mask.getDownloadURL({
        'region':square,
        'dimensions':'512x512',
        'crs':'EPSG:4326',
        'format': 'GEO_TIFF'
    })
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        raise r.raise_for_status()
    with open(filename, 'wb') as out_file:
        out_file.write(r.content)
    
# DOWNLOAD IMAGES
points = centroids.aggregate_array('.geo').getInfo() 
with multiprocessing.Pool(25) as pool:
    results = list(tqdm(pool.imap(getResult, list(enumerate(points))), total=len(points)))
    pool.close()
    pool.join()

  3%|▎         | 314/9911 [01:39<29:59,  5.33it/s]  